# 📘 Análise e Implementação de Regras de Associação no Fantasy Premier League (FPL)

Este notebook aplica o algoritmo Apriori sobre o dataset `merged_gw_processed.csv`, contendo dados detalhados de desempenho dos jogadores na época 2024/2025 da Fantasy Premier League (FPL).  
O objetivo é identificar padrões relevantes no comportamento e performance dos jogadores ao longo das Gameweeks, criando regras de associação úteis para análise descritiva e potencial suporte à tomada de decisões.

O Fantasy Premier League é um jogo onde milhões de utilizadores constroem equipas de futebol baseadas em jogadores reais da Premier League. A performance real dos atletas traduz-se em pontos utilizados para competir em ligas privadas ou globais.  
Assim, compreender relações entre desempenhos, condições de jogo e eventos específicos pode ajudar a derivar conhecimento útil sobre tendências e sinergias dentro do jogo.

Este notebook assume que o dataset já foi limpo e discretizado previamente na fase de EDA, sendo agora adequado para aplicação de algoritmos de mineração de padrões frequentes.

## 🎯 Objetivos de Negócio

A aplicação de regras de associação neste contexto pretende alcançar:

### ✔️ Identificar padrões frequentes no desempenho dos jogadores
Exemplos:
- Jogadores que marcam golos também têm tendência a receber bónus.
- Jogadores baratos com elevado "creativity_tier" tendem a obter pontos acima da média.

### ✔️ Descobrir combinações de atributos que ocorrem conjuntamente
Como:
- "starts=1" + "value_tier=cheap" + "points_tier=high".

### ✔️ Gerar regras de associação que permitam interpretar ligações entre eventos do jogo
Tais como:
- Jogadores que fazem assistências também tendem a ter maior "influence_tier".

### ✔️ Criar insights para análises descritivas e exploratórias
Permitindo:
- Melhor caracterização dos tipos de jogadores.
- Suporte a futuras análises preditivas.
- Identificação de padrões recorrentes ao longo da época.

Este notebook foca-se na extração e avaliação destas regras através do algoritmo **Apriori**.

## 📥 Carregamento do Dataset Processado (Python)

In [6]:
import pandas as pd

df = pd.read_csv("data/merged_gw_processed.csv")
df.head()

,assists,bonus,clean_sheets,goals_conceded,goals_scored,own_goals,penalties_missed,penalties_saved,red_cards,starts,...,xP_Medium,bps_High,bps_Low,bps_Medium,Points_High,Points_Low,Points_Medium,Value_Budget,Value_Mid,Value_Premium
0,0,0,0,1,0,0,0,0,0,1,...,1,0,0,1,0,0,1,0,1,0
1,0,0,0,1,0,0,0,0,0,1,...,0,0,1,0,0,1,0,0,0,1
2,0,0,0,1,0,0,0,0,0,1,...,1,1,0,0,0,0,1,1,0,0
3,0,0,0,1,0,0,0,0,0,1,...,0,1,0,0,0,0,1,0,0,1
4,0,0,0,1,0,0,0,0,0,1,...,0,0,0,1,0,1,0,0,1,0


## 📊 Estrutura Final do Dataset para Apriori

Antes de aplicar o Apriori, é necessário confirmar que o dataset está num formato adequado:
- Todas as colunas categóricas estão discretizadas.
- Não existem colunas numéricas contínuas.
- Não existem valores nulos.
- As features representam atributos binários ou categorias discretas.

A seguir apresenta-se a estrutura final do dataset.

In [7]:
df.info() 
df.describe(include='all')

## Find values with data above 1 in all dataset
for column in df.columns:
    if df[column].dtype != 'object':  # Check if the column is not of object type
        count_above_one = (df[column] > 1).sum()
        print(f"Column '{column}' has {count_above_one} values above 1.")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6347 entries, 0 to 6346
Data columns (total 70 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   assists                            6347 non-null   int64
 1   bonus                              6347 non-null   int64
 2   clean_sheets                       6347 non-null   int64
 3   goals_conceded                     6347 non-null   int64
 4   goals_scored                       6347 non-null   int64
 5   own_goals                          6347 non-null   int64
 6   penalties_missed                   6347 non-null   int64
 7   penalties_saved                    6347 non-null   int64
 8   red_cards                          6347 non-null   int64
 9   starts                             6347 non-null   int64
 10  was_home                           6347 non-null   int64
 11  yellow_cards                       6347 non-null   int64
 12  pos_DEF             

## 🧠 Algoritmo Apriori

O Apriori é um algoritmo clássico de mineração de padrões frequentes.  
Opera segundo dois princípios fundamentais:

### ✔️ 1. Geração de Itemsets Frequentes
Identifica combinações de atributos que ocorrem conjuntamente acima de um limiar mínimo de suporte.

### ✔️ 2. Geração de Regras de Associação
A partir dos itemsets frequentes, cria regras do tipo:

A → B

que representam:

> sempre que o conjunto A ocorre, o conjunto B tende também a ocorrer.

As métricas avaliadas serão:
- **Support** → frequência conjunta  
- **Confidence** → probabilidade condicional  
- **Lift** → força da associação comparada ao acaso  

Este notebook implementa o Apriori utilizando a biblioteca `mlxtend`.


## 🚀 Primeira Iteração do Algoritmo Apriori

Nesta secção, o notebook aplica pela primeira vez o algoritmo **Apriori** ao dataset já transformado e preparado.  
O objetivo desta etapa é identificar **padrões frequentes simples**, de tamanho até 3 itens, que permitam compreender quais combinações de atributos ocorrem com maior frequência entre os jogadores na Fantasy Premier League.

Esta primeira iteração funciona como uma fase exploratória, permitindo:
- Avaliar a densidade das combinações frequentes.
- Verificar se os níveis de discretização e o one-hot encoding estão a produzir padrões coerentes.
- Obter itemsets iniciais que servirão de base para a geração de regras (na fase seguinte).

Foram utilizados:
- `min_support = 0.03`: apenas combinações presentes em pelo menos 3% dos registos.
- `max_len = 3`: limitação intencional para captar apenas combinações simples.
- `use_colnames = True`: para garantir legibilidade dos resultados.

Após a obtenção de todos os itemsets, foram filtrados aqueles contendo **exatamente 2 ou 3 itens**, considerados os mais relevantes para análise de padrões e potenciais regras.

In [8]:
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings("ignore")

# Aplicação do Apriori – primeira iteração
frequent_items = apriori(df, min_support=0.03, max_len=3, use_colnames=True)

num_itemsets = len(frequent_items)
print(f'Number of itemsets: {num_itemsets}')

itemsets_2_3 = frequent_items[
    frequent_items['itemsets'].apply(lambda x: 2 <= len(x) <= 3)
]

itemsets_2_3

Number of itemsets: 7111


,support,itemsets
64,0.050733,"(assists, goals_conceded)"
65,0.067591,"(starts, assists)"
66,0.040334,"(was_home, assists)"
67,0.051520,"(pos_MID, assists)"
68,0.078620,"(assists, save_null)"
...,...,...
7106,0.049315,"(Value_Mid, Points_Low, bps_Medium)"
7107,0.037025,"(Points_Low, bps_Medium, Value_Premium)"
7108,0.054829,"(Points_Medium, bps_Medium, Value_Budget)"
7109,0.053726,"(Value_Mid, Points_Medium, bps_Medium)"


### Avaliação dos Padrões Frequentes (Primeira Iteração)

A primeira execução do algoritmo Apriori gerou um total de **7111 itemsets**, dos quais uma parte significativa corresponde a combinações com 2 ou 3 atributos, exatamente o objetivo para esta fase exploratória.

#### Interpretação Geral dos Resultados

De forma geral, os padrões obtidos apresentam:
- **Coerência contextual** (ex.: `starts` surge frequentemente combinado com métricas ofensivas como `assists`).
- **Coerência posicional** (ex.: `pos_MID` aparece em itemsets com `assists`, confirmando relações esperadas).
- **Coerência táctica/contextual** (ex.: `was_home` aparece combinado com `assists`, refletindo vantagem de jogar em casa).
- **Padrões derivados da discretização** (ex.: `Value_Mid`, `bps_Medium`, `Points_Medium` surgem frequentemente combinados, caracterizando perfis de jogadores).

Os itemsets derivados das colunas discretizadas são particularmente interessantes, sendo capazes de revelar **perfis de jogadores com valor semelhante, contribuição BPS semelhante e nível de pontuação semelhante**, o que é extremamente útil para gerar regras de associação.

#### Exemplos de Padrões Relevantes que Surgiram

- `(starts, assists)` indica que jogadores titulares contribuem ofensivamente.
- `(was_home, assists)` sugere maior probabilidade de assistências em jogos disputados em casa.
- `(pos_MID, assists)` confirma que os médios são os principais responsáveis por assistências.
- Itemsets com tiers, como `(Value_Premium, bps_Medium, Points_Medium)`, mostram que jogadores caros nem sempre correspondem a desempenhos elevados, um resultado surpreendente e potencialmente valioso.

No geral, os padrões são coerentes, informativos e suficientes para avançar para regras de associação.

## 🚀 Segunda Iteração do Algoritmo Apriori

Após a primeira interação com o algoritmo Apriori, verificou-se que:

- Foram obtidos mais de **7000 itemsets** (com combinações de 2 a 3 itens).
- Muitas associações eram demasiado óbvias ou triviais, como:
  - (MID, assists)
  - (bps_Medium, Value_Mid)
  - (Value_Budget, Points_Low)

Apesar de úteis para compreender a estrutura dos dados, estas regras ainda não capturavam padrões realmente fortes ou surpreendentes que possam gerar insights profundos sobre o comportamento dos jogadores no FPL.

Assim, nesta segunda iteração procedemos a ajustes estratégicos:

- Aumentar o suporte mínimo para focar associações mais consistentes.
- Extrair regras de associação (confidence, lift), ausentes na primeira fase.
- Filtrar regras mais fortes (lift > 1.1).
- Manter apenas regras com 2 a 3 itens para coerência analítica.

O objetivo é refinar os padrões descobertos e aproximar a análise de relações realmente relevantes para tomada de decisão.

### Implementação — Segunda Iteração (Ajuste de Parâmetros do Apriori)

#### 1. Aumento do min_support (0.03 → 0.05)
O objetivo é eliminar padrões demasiado raros e concentrar a atenção em comportamentos que se manifestam de forma consistente ao longo das Gameweeks. Isto reduz ruído e torna as regras mais robustas.

#### 2. Extração de Regras de Associação
Nesta fase introduzimos:
- **Support**
- **Confidence**
- **Lift**

A primeira iteração apenas extraía itemsets, não regras.

O parâmetro `metric="lift"` + `min_threshold=1.1` permite identificar relações estatisticamente mais úteis.

#### 3. Filtragem para Regras de 2 a 3 Itens
Mantém coerência com a análise anterior e evita:
- regras triviais de 1 item  
- regras demasiado complexas (4+ itens)

O objetivo é focar as associações mais interpretáveis e relevantes.

In [9]:
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings("ignore")

# Segunda iteração do Apriori — suporte mais alto e extração de regras
frequent_items_v2 = apriori(
    df,
    min_support=0.05,     # antes 0.03 — mais rigoroso nesta fase
    max_len=3,
    use_colnames=True
)

print(f"Número de itemsets após segunda iteração: {len(frequent_items_v2)}")

# Extração de regras de associação
rules_v2 = association_rules(
    frequent_items_v2,
    metric="lift",
    min_threshold=1.1     # foco em relações mais fortes
)

# Filtrar regras com total de 2 ou 3 itens
rules_v2["num_items"] = rules_v2["antecedents"].apply(len) + rules_v2["consequents"].apply(len)
rules_v2_filtered = rules_v2[rules_v2["num_items"].between(2, 3)]

# Mostrar top regras por lift
rules_v2_filtered.sort_values("lift", ascending=False).head(20)

Número de itemsets após segunda iteração: 4840


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,num_items
838,"(bonus, threat_(8.0, 120.0])",(goals_scored),0.074996,0.086813,0.061604,0.821429,9.462082,1.0,0.055093,5.113849,0.966823,0.614780,0.804453,0.765524,3
843,(goals_scored),"(bonus, threat_(8.0, 120.0])",0.086813,0.074996,0.061604,0.709619,9.462082,1.0,0.055093,3.185482,0.979334,0.614780,0.686076,0.765524,3
849,(goals_scored),"(bonus, expected_goal_involvements_High)",0.086813,0.074523,0.057665,0.664247,8.913266,1.0,0.051195,2.756420,0.972208,0.556231,0.637211,0.719016,3
844,"(bonus, expected_goal_involvements_High)",(goals_scored),0.074523,0.086813,0.057665,0.773784,8.913266,1.0,0.051195,4.036800,0.959298,0.556231,0.752279,0.719016,3
826,"(bonus, ict_index_High)",(goals_scored),0.088546,0.086813,0.062392,0.704626,8.116630,1.0,0.054705,3.091634,0.961975,0.552301,0.676546,0.711660,3
831,(goals_scored),"(bonus, ict_index_High)",0.086813,0.088546,0.062392,0.718693,8.116630,1.0,0.054705,3.240073,0.960149,0.552301,0.691365,0.711660,3
815,(goals_scored),"(bonus, save_null)",0.086813,0.096108,0.062392,0.718693,7.477945,1.0,0.054048,3.213189,0.948626,0.517647,0.688783,0.683937,3
812,"(bonus, save_null)",(goals_scored),0.096108,0.086813,0.062392,0.649180,7.477945,1.0,0.054048,2.603011,0.958382,0.517647,0.615829,0.683937,3
850,"(bonus, xP_High)",(goals_scored),0.083819,0.086813,0.054199,0.646617,7.448412,1.0,0.046922,2.584126,0.944948,0.465494,0.613022,0.635468,3
855,(goals_scored),"(bonus, xP_High)",0.086813,0.083819,0.054199,0.624319,7.448412,1.0,0.046922,2.438723,0.948046,0.465494,0.589949,0.635468,3


### 📊 Interpretação dos Resultados da Segunda Iteração

A análise revelou um conjunto consistente de regras extremamente fortes,
especialmente envolvendo:

#### ⭐ Padrão mais presente:

**Players que marcaram golos → têm ameaça alta e recebem bónus**

Exemplos:
- `(goals_scored) → (threat_(8.0,120.0], bonus)`
  - conf = 0.71
  - lift = 9.46

- `(threat_(8.0,120.0], bonus) → goals_scored`
  - conf = 0.82
  - lift = 9.46

Isto é estatisticamente fortíssimo:  
> Jogadores com ameaça alta + bónus estão **9× mais associados** a marcar golos do que seria esperado.

### 🔥 Outro padrão forte:

**Goals Scored → ICT alto + Bonus**

- lift ≈ 8.1  
- Confiança ~70%

Este padrão confirma coerência no dataset:
> Jogadores que marcam golos quase sempre geram impacto elevado (ICT High) e acumulam bónus.

### ⚽ Padrão ofensivo global extraído:

Jogadores com:
- *Threat high*
- *Influence high*
- *Expected goal involvements high*
- *xP high*

→ **Têm probabilidade muito acima da média de marcar golos e obter bónus**.

### 📌 Detalhe interessante:

Regras inversas também aparecem fortes:
- `(bonus, xP_High) → goals_scored`  
- `(goals_scored, xP_High) → bonus`

Mostrando uma relação circular de reforço entre:
- rendimento previsto (xP)
- performance real (golos)
- pontuação adicional (bonus)

### 📌 Avaliação geral:

As regras revelam **coerência interna** e **qualidade estatística elevada**,
com lifts entre **7 e 9**, o que é excecional.

Estes resultados validam:
- A discretização foi bem feita  
- O dataset contém estrutura  
- As métricas avançadas estão a ajudar a isolar padrões reais  

### 🔄 Iteração 3 — Otimização Final das Regras e Extração de Padrões Mais Relevantes

#### 🎯 Objetivo
Refinar ainda mais o conjunto de regras, focando exclusivamente em:
- Regras **curtas** (2–3 itens),
- Regras **altamente confiáveis** (confidence ≥ 0.75),
- Regras **estatisticamente fortes** (lift ≥ 1.2),
- Remover redundância e sobreposição.

#### 📌 1. Filtragem Avançada de Regras (Support, Confidence, Lift)

In [10]:
filtered_rules = rules_v2[
    (rules_v2['antecedents'].apply(len).between(1,2)) &
    (rules_v2['consequents'].apply(len).between(1,2)) &
    (rules_v2['confidence'] >= 0.75) &
    (rules_v2['lift'] >= 1.2)
].sort_values(by='lift', ascending=False)

filtered_rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,num_items
838,"(bonus, threat_(8.0, 120.0])",(goals_scored),0.074996,0.086813,0.061604,0.821429,9.462082,1.0,0.055093,5.113849,0.966823,0.614780,0.804453,0.765524,3
844,"(bonus, expected_goal_involvements_High)",(goals_scored),0.074523,0.086813,0.057665,0.773784,8.913266,1.0,0.051195,4.036800,0.959298,0.556231,0.752279,0.719016,3
852,"(xP_High, goals_scored)",(bonus),0.070585,0.103513,0.054199,0.767857,7.417944,1.0,0.046892,3.861788,0.930899,0.452037,0.741053,0.645725,3
822,"(played_60+, goals_scored)",(bonus),0.073736,0.103513,0.055459,0.752137,7.266076,1.0,0.047827,3.616859,0.931024,0.455369,0.723517,0.643953,3
3613,(goals_scored),"(threat_(8.0, 120.0], Points_High)",0.086813,0.147944,0.085237,0.981851,6.636645,1.0,0.072394,46.948291,0.930063,0.570074,0.978700,0.778998,3
3559,(goals_scored),"(threat_(8.0, 120.0], influence_High)",0.086813,0.148889,0.085237,0.981851,6.594507,1.0,0.072312,46.896203,0.929008,0.566492,0.978676,0.777169,3
3607,(goals_scored),"(threat_(8.0, 120.0], bps_High)",0.086813,0.154404,0.083977,0.967332,6.264956,1.0,0.070572,25.884644,0.920273,0.534068,0.961367,0.755605,3
3565,(goals_scored),"(influence_High, expected_goal_involvements_High)",0.086813,0.155507,0.078462,0.903811,5.812047,1.0,0.064962,8.779545,0.906652,0.478846,0.886099,0.704185,3
3637,(goals_scored),"(Points_High, expected_goal_involvements_High)",0.086813,0.155664,0.078462,0.903811,5.806164,1.0,0.064949,8.777907,0.906462,0.478386,0.886078,0.703930,3
742,(assists),"(creativity_High, Points_High)",0.079092,0.139594,0.062707,0.792829,5.679553,1.0,0.051666,4.153116,0.894693,0.402020,0.759217,0.621019,3


#### 📌 2. Remoção de Regras Redundantes

In [11]:
def remove_redundancy(rules_df):
    unique_rules = []
    seen = set()

    for _, row in rules_df.iterrows():
        rule = (frozenset(row['antecedents']), frozenset(row['consequents']))
        if rule not in seen:
            seen.add(rule)
            unique_rules.append(row)
    
    return pd.DataFrame(unique_rules)

clean_rules = remove_redundancy(filtered_rules)
clean_rules.head(20)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,num_items
838,"(bonus, threat_(8.0, 120.0])",(goals_scored),0.074996,0.086813,0.061604,0.821429,9.462082,1.0,0.055093,5.113849,0.966823,0.614780,0.804453,0.765524,3
844,"(bonus, expected_goal_involvements_High)",(goals_scored),0.074523,0.086813,0.057665,0.773784,8.913266,1.0,0.051195,4.036800,0.959298,0.556231,0.752279,0.719016,3
852,"(xP_High, goals_scored)",(bonus),0.070585,0.103513,0.054199,0.767857,7.417944,1.0,0.046892,3.861788,0.930899,0.452037,0.741053,0.645725,3
822,"(played_60+, goals_scored)",(bonus),0.073736,0.103513,0.055459,0.752137,7.266076,1.0,0.047827,3.616859,0.931024,0.455369,0.723517,0.643953,3
3613,(goals_scored),"(threat_(8.0, 120.0], Points_High)",0.086813,0.147944,0.085237,0.981851,6.636645,1.0,0.072394,46.948291,0.930063,0.570074,0.978700,0.778998,3
3559,(goals_scored),"(threat_(8.0, 120.0], influence_High)",0.086813,0.148889,0.085237,0.981851,6.594507,1.0,0.072312,46.896203,0.929008,0.566492,0.978676,0.777169,3
3607,(goals_scored),"(threat_(8.0, 120.0], bps_High)",0.086813,0.154404,0.083977,0.967332,6.264956,1.0,0.070572,25.884644,0.920273,0.534068,0.961367,0.755605,3
3565,(goals_scored),"(influence_High, expected_goal_involvements_High)",0.086813,0.155507,0.078462,0.903811,5.812047,1.0,0.064962,8.779545,0.906652,0.478846,0.886099,0.704185,3
3637,(goals_scored),"(Points_High, expected_goal_involvements_High)",0.086813,0.155664,0.078462,0.903811,5.806164,1.0,0.064949,8.777907,0.906462,0.478386,0.886078,0.703930,3
742,(assists),"(creativity_High, Points_High)",0.079092,0.139594,0.062707,0.792829,5.679553,1.0,0.051666,4.153116,0.894693,0.402020,0.759217,0.621019,3


#### 📌 3. Ranking Final das Regras

Selecionamos as regras mais úteis com base em lift × confidence.

In [12]:
clean_rules['score'] = clean_rules['lift'] * clean_rules['confidence']
top_rules = clean_rules.sort_values(by='score', ascending=False)

top_rules.head(15)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,num_items,score
838,"(bonus, threat_(8.0, 120.0])",(goals_scored),0.074996,0.086813,0.061604,0.821429,9.462082,1.0,0.055093,5.113849,0.966823,0.614780,0.804453,0.765524,3,7.772424
844,"(bonus, expected_goal_involvements_High)",(goals_scored),0.074523,0.086813,0.057665,0.773784,8.913266,1.0,0.051195,4.036800,0.959298,0.556231,0.752279,0.719016,3,6.896945
3613,(goals_scored),"(threat_(8.0, 120.0], Points_High)",0.086813,0.147944,0.085237,0.981851,6.636645,1.0,0.072394,46.948291,0.930063,0.570074,0.978700,0.778998,3,6.516197
3559,(goals_scored),"(threat_(8.0, 120.0], influence_High)",0.086813,0.148889,0.085237,0.981851,6.594507,1.0,0.072312,46.896203,0.929008,0.566492,0.978676,0.777169,3,6.474825
3607,(goals_scored),"(threat_(8.0, 120.0], bps_High)",0.086813,0.154404,0.083977,0.967332,6.264956,1.0,0.070572,25.884644,0.920273,0.534068,0.961367,0.755605,3,6.060293
852,"(xP_High, goals_scored)",(bonus),0.070585,0.103513,0.054199,0.767857,7.417944,1.0,0.046892,3.861788,0.930899,0.452037,0.741053,0.645725,3,5.695921
822,"(played_60+, goals_scored)",(bonus),0.073736,0.103513,0.055459,0.752137,7.266076,1.0,0.047827,3.616859,0.931024,0.455369,0.723517,0.643953,3,5.465083
3547,(goals_scored),"(ict_index_High, Points_High)",0.086813,0.187490,0.086813,1.000000,5.333613,1.0,0.070536,inf,0.889752,0.463025,1.000000,0.731513,3,5.333613
3565,(goals_scored),"(influence_High, expected_goal_involvements_High)",0.086813,0.155507,0.078462,0.903811,5.812047,1.0,0.064962,8.779545,0.906652,0.478846,0.886099,0.704185,3,5.252993
6817,(pos_GK),"(played_60+, creativity_Low)",0.066803,0.171262,0.063337,0.948113,5.536039,1.0,0.051896,15.972041,0.878020,0.362489,0.937391,0.658969,3,5.248792


#### 📌 4. Análise dos Padrões Descobertos

##### 1. **Padrões esperados**

Algumas regras identificadas refletem relações já conhecidas no contexto do FPL e do futebol real, funcionando como uma validação de que o algoritmo está a capturar corretamente as dinâmicas do jogo.

**🔹 Padrão 1 — ``(bonus, threat alto) → goals_scored``**

É natural que jogadores com threat elevado sejam responsáveis por grande volume ofensivo (remates, criação de oportunidades).
Juntar isso ao bónus, que no FPL muitas vezes favorece jogadores que marcam golos, torna esta regra previsível.

Porque é esperado:

- Jogadores com alto threat tendem a chutar muito.

- Marcar golos impulsiona fortemente os pontos BPS → logo, receber bónus é comum.

- Lift ~9 confirma fortíssima associação, mas dentro do esperado.

**🔹 Padrão 2 — ``(played 60+, goals_scored) → bonus``**

Bónus no FPL geralmente favorece:

- jogadores que marcam golo,

- jogadores que jogam muitos minutos.

Porque é esperado:

- O sistema BPS beneficia golos.

- Jogar >60 minutos aumenta impacto no BPS e reduz concorrência.

- Jogador que marca e joga muito quase sempre recebe bónus.

**🔹 Padrão 3 — ``goals_scored → (bps_High, ict_index_High, influence_High)``**

O ICT Index e o BPS foram criados precisamente para medir performance ofensiva.

Porque é esperado:

Marcar golos aumenta automaticamente:

- Threat,

- Influence,

- ICT,

- BPS → muitos destes itens aparecem como consequentes.

#### 2. **Padrões Interessantes e Inesperados**

Algumas regras são surpreendentes e revelam tendências pouco óbvias, que podem ser úteis para insights ou recomendações estratégicas.

**⭐ Padrão 1 — ``pos_GK → (played_60+, creativity_Low)``**

Embora pareça lógico que GR tenham baixa criatividade, o que surpreende é a força da regra, com confiança acima de 0.95 e lift >5.

Porque é inesperado:

- O algoritmo captou os guarda-redes como um cluster extremamente homogéneo.

- Criatividade baixa é esperada, mas o nível de força da associação (lift >5) indica que é praticamente determinístico no dataset.

- Mostra que métricas de ataque podem ser quase inúteis para GR, podendo ser eliminadas em certas análises futuras.

**⭐ Padrão 2 — ``assists → (creativity_High, Points_High)``**

Esta regra é intuitiva, mas a força da relação é surpreendentemente alta (lift ~5.68).

Porque é inesperado:

- Esperava-se creativity→assists, mas não o inverso tão forte.

- Isto sugere que jogadores que já deram assistências têm consistentemente performances mais completas (não apenas uma jogada isolada).

- Pode refletir estilos de jogo de certas posições (ex: laterais ofensivos, médios criativos).

**⭐ Padrão 3 — goals_scored → (expected_goal_involvements_High + Points_High)**

A presença de XGI alto é normal, mas o lift >5.8 mostra que quem marca golo realmente está sempre nos patamares mais altos de XGI.

Porque é inesperado:

- XGI normalmente prevê golos, mas aqui vemos o inverso:

- marcar golo é praticamente garantia que o jogador estava em XGI alto.

- Indica que há poucos “outliers” que marcam golos com XGI baixo (ex: finalizações isoladas).

**⭐ Padrão 4 — ``(bonus, xP_High) → goals_scored``**

xP ("expected points") não depende diretamente de golos num determinado jogo, é acumulativo e inclui probabilidade.

Porque é inesperado:

- O algoritmo mostra que jogadores com históricos bons de xP tendem a marcar quando também recebem bónus.

- Ou seja, jogadores em boa forma acumulada tendem a ter picos de rendimento previsíveis.

- Isto sugere consistência → útil para forecasting.

**⭐ Padrão 5 — ``pos_GK → (starts + expected_goal_involvements_Low)``**

Aqui a combinação é relevante: GR não só começam quase sempre os jogos, como raramente têm XGI alto, o que já era esperado, MAS o lift de ~4.9 mostra que é um padrão em absoluto.

Porque é inesperado:

- Mostra que os GR são um cluster totalmente separado do resto dos jogadores em métricas ofensivas.

- Útil insight para segmentação: GR podem ser analisados em pipelines separados no futuro.